# Introduction to Temporal Windows

Like a time-series rolling window operation, with a head and a tail

A temporal window "modifies" an index or pipeline from providing its original mapping from query to result, so it lives in the 'modifications' area of the API. The code below is a copy of the source code of PyEarthTools, to show exactly what it does and how it works.

In [1]:
import functools
import xarray as xr
import pyearthtools.pipeline
class TemporalWindow(pyearthtools.pipeline.controller.PipelineIndex):
    '''
    The purpose of this class is to provide the ability to perform
    sequence-to-sequence modelling from an data accessor or pipeline
    that was designed to produce single time steps (i.e. single samples).

    The temporal window allows the specification of the 'back window'
    and the 'forward window', and will produce a binary branch.

    For example, if the time steps are hourly, and the base pipeline
    can produce hours 1, 2, 3 ... 10; then this Temporal Window can
    be used to produce sequence pairs like:
         [1,2,3], [4], 
         [2,3,4], [5],
         ...
         [7,8,9], [10]

    or like:
         [1,2], [3,4,5],
         [2,3], [4,5,6],
         ...
         [6,7], [8,9,10]

    This provides a simpler interface than the TemporalRetrieval which
    is a more general alternative.

    The window offsets are calculated not using positional indexing, but 
    using calculated date-times based on the reference time and the specified
    timedelta to calculate each required index exactly. The handling of missing
    data is left to the underlying pipeline response to the retrieval of the
    calculated datetime.

    The resultant sequences may be left unmerged (i.e. a list of retrieved
    results for each timetime) or merged (e.g. into an xarray along the time
    dimension). The default behaviour is to merge along the time dimension.

    A custom merge method may be specified.
    '''

    def __init__(self, *, prior_indexes, posterior_indexes, timedelta, merge_method=None):
        """
        Args:
            prior_indexes: Multiplied by the timedelta then applied to the reference date
            posterior_indexes: Multiplied by the timedelta then applied to the reference date
            timedelta: Typically the time step of the underlying data
            merge_method: How to merge samples into a combined object


        Examples:

        >>> TemporalWindow(prior_indexes=[-3,-2,-1], posterior_indexes=[0], timedelta=timedelta, merge_method=merge_method)

        (assuming xarray data) will result in a tuple of two  datasets, the first with a time coordinate dimension of 3 time steps and the
        second with a time coordinate dimension of 1 time step.

        """
        self.prior_indexes = prior_indexes
        self.posterior_indexes = posterior_indexes
        self.timedelta = timedelta
        self.merge_method = merge_method

    def __getitem__(self, date_of_interest):
        date_of_interest = pyearthtools.data.Petdt(date_of_interest)

        prior_i = [i * self.timedelta for i in self.prior_indexes]
        posterior_i = [i * self.timedelta for i in self.posterior_indexes]

        prior = [self.parent_pipeline()[str(date_of_interest + delta)] for delta in prior_i]
        posterior = [self.parent_pipeline()[str(date_of_interest + delta)] for delta in posterior_i]

        if self.merge_method:
            prior = self.merge_method(prior)
            posterior = self.merge_method(posterior)

        return prior, posterior
        


In [11]:
import os
from pathlib import Path

# The default directory for cached download data is the user home directory
os.environ['PETPROJECT'] = os.path.expanduser("~") + '/petcache'
workdir = Path(os.environ['PETPROJECT'])
# print(workdir)

import pathlib
import xarray as xr
from pathlib import Path
import time

import pyearthtools.data.archive
import pyearthtools.tutorial
import pyearthtools.pipeline


In [20]:
# This is just under 3GB of data
accessor = pyearthtools.data.download.weatherbench.WB2ERA5(
        variables=["2m_temperature", "u", "v", "geopotential", "vorticity"],
        level=[850],
        download_dir=workdir / "download",
        license_ok=True,
    )

In [21]:
sample = accessor['2010-01-01T00']
sample

<xarray.Dataset> Size: 42kB
Dimensions:              (time: 1, longitude: 64, latitude: 32, level: 1)
Coordinates:
  * time                 (time) datetime64[ns] 8B 2010-01-01
  * longitude            (longitude) float64 512B 0.0 5.625 ... 348.8 354.4
  * latitude             (latitude) float64 256B -87.19 -81.56 ... 81.56 87.19
  * level                (level) int64 8B 850
Data variables:
    2m_temperature       (time, longitude, latitude) float32 8kB dask.array<chunksize=(1, 64, 32), meta=np.ndarray>
    u_component_of_wind  (time, level, longitude, latitude) float32 8kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>
    v_component_of_wind  (time, level, longitude, latitude) float32 8kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>
    geopotential         (time, level, longitude, latitude) float32 8kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>
    vorticity            (time, level, longitude, latitude) float32 8kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>

In [22]:
timedelta = pyearthtools.data.time.TimeDelta((1, "day"))
merge_method = functools.partial(xr.concat, dim='time')

In [24]:
data_pipeline = pyearthtools.pipeline.Pipeline(
    accessor,
    pyearthtools.data.transforms.coordinates.StandardLongitude(type="-180-180"),     
    # pyearthtools.pipeline.modifications.TemporalRetrieval(
    #     concat=True, samples=((0, 1), (6, 1, 6)) # Input = 1 sample from time T=0 hours. Output = T+6,+12,+18,+24
    # ),     
    TemporalWindow(prior_indexes=[-3,-2,-1], posterior_indexes=[0], timedelta=timedelta, merge_method=merge_method),
    sampler=pyearthtools.pipeline.samplers.Default(),
    iterator=pyearthtools.pipeline.iterators.DateRange(1980, 2016, interval='6 hours')
)

In [25]:
doi = '20100101T0000'
sample = data_pipeline[doi]

/Users/munin/dev/proj/PyEarthTools/packages/data/src/pyearthtools/data/indexes/_indexes.py:789: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(


In [26]:
len(sample)

2

In [27]:
sample[0]

<xarray.Dataset> Size: 124kB
Dimensions:              (time: 3, longitude: 64, latitude: 32, level: 1)
Coordinates:
  * time                 (time) datetime64[ns] 24B 2009-12-29 ... 2009-12-31
  * longitude            (longitude) float64 512B -180.0 -174.4 ... 168.8 174.4
  * latitude             (latitude) float64 256B -87.19 -81.56 ... 81.56 87.19
  * level                (level) int64 8B 850
Data variables:
    2m_temperature       (time, longitude, latitude) float32 25kB dask.array<chunksize=(1, 64, 32), meta=np.ndarray>
    u_component_of_wind  (time, level, longitude, latitude) float32 25kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>
    v_component_of_wind  (time, level, longitude, latitude) float32 25kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>
    geopotential         (time, level, longitude, latitude) float32 25kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>
    vorticity            (time, level, longitude, latitude) float32 25kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>

In [28]:
sample[1]

<xarray.Dataset> Size: 42kB
Dimensions:              (time: 1, longitude: 64, latitude: 32, level: 1)
Coordinates:
  * time                 (time) datetime64[ns] 8B 2010-01-01
  * longitude            (longitude) float64 512B -180.0 -174.4 ... 168.8 174.4
  * latitude             (latitude) float64 256B -87.19 -81.56 ... 81.56 87.19
  * level                (level) int64 8B 850
Data variables:
    2m_temperature       (time, longitude, latitude) float32 8kB dask.array<chunksize=(1, 64, 32), meta=np.ndarray>
    u_component_of_wind  (time, level, longitude, latitude) float32 8kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>
    v_component_of_wind  (time, level, longitude, latitude) float32 8kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>
    geopotential         (time, level, longitude, latitude) float32 8kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>
    vorticity            (time, level, longitude, latitude) float32 8kB dask.array<chunksize=(1, 1, 64, 32), meta=np.ndarray>